# 2 — Retrieval and Generation

Now we ask questions of the index we built in notebook 1.

Four things happen, and all four are used in real systems:

```
question
   |
   v
search          retrieval.search            vector search over the whole index
   |                                        wide: tuned for recall
   v
rerank          (inside search)             a cross-encoder reorders ~30
   |                                        narrow: tuned for precision
   v
filter          filters={...}               restrict what search may look at
   |                                        runs inside the database
   v
table rows      retrieval.with_table_rows   a summary matched? fetch its rows
   |
   v
answer          retrieval.answer            fill a token budget, generate
```

**This does NOT parse or embed a corpus.** Everything here runs per question, in
under a second. The index was built in notebook 1.

The code lives in the `rag` package, same as notebook 1. Each cell here is a few
lines — enough to see what a step does. Open `rag/retrieval.py` if you want the
details.

| § | What |
|---|---|
| 0 | Setup |
| 1 | Search and rerank |
| 2 | Filtering with metadata |
| 3 | Table rows |
| 4 | Build the prompt and answer |

---
## 0. Setup

In [ ]:
# Where you've uploaded this project's files in your Drive — the rag/ package,
# requirements.txt, and pdfs/. Adjust the path if you put it somewhere else.
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/05-rag'
%cd {PROJECT_DIR}
%pip install -q -r requirements.txt

In [ ]:
# Secrets come from Colab's own manager, not a .env file — click the key icon
# in the left sidebar, add OPENAI_API_KEY and PINECONE_API_KEY once, and they
# persist across sessions without ever being typed into a cell or saved in the
# notebook itself.
#
# MUST run before `from rag import ...` below — config.py reads every setting
# at import time, so setting these after the import has no effect.
import os
from google.colab import userdata

for name in ("OPENAI_API_KEY", "PINECONE_API_KEY"):
    os.environ[name] = userdata.get(name)

In [ ]:
import pandas as pd

from rag import retrieval
from rag.config import RERANK_MODEL
from rag.index import list_rerank_models, open_index, read_manifest

# Checks that this notebook's settings match how the index was built. It raises
# rather than warns: search an index with the wrong embedding model and you get
# results back, with scores that look fine, that are meaningless.
manifest = read_manifest()
index = open_index()
DIMS = manifest["embed_dims"]

print(f"index     : {manifest['index_name']} ({DIMS}d)")
print(f"embedding : {manifest['embed_model']}")
print(f"reranker  : {RERANK_MODEL}")
print(f"documents : {list(manifest['documents'])}")

---
## 1. Search and rerank

### How dense search works

At ingestion, the embedding model read each chunk and produced a vector — 1536
numbers.

At query time, the same model reads your question and produces a vector the same way.

The index returns the chunks whose vectors are closest to the question's vector.

That is all it does. It compares two vectors.

### Why that is not accurate enough on its own

**The chunk's vector was computed before your question existed.**

A chunk of 1000 words might cover five different things. All of it becomes one
vector, and the model had to decide what to keep without knowing what you would ask.
So the vector represents the chunk's topic — not whether it answers you.

**One word cannot change the vector much.**

Every word contributes to the average. A word that reverses the meaning is one out of
a thousand.

    What are the inclusion criteria?
    What are the exclusion criteria?

Same words, opposite meaning. Both land in almost the same place, because those two
words appear in identical sentences everywhere:

    The inclusion criteria are listed in Section 5.
    The exclusion criteria are listed in Section 5.

The model learns words from the sentences around them. These two always have the same
sentences around them, so it treats them as similar.

So a passage about exclusions ranks highly for both questions. For one of them, that
is wrong.

### What the reranker does

It takes your question and one chunk and feeds **both together** into the model as a
single input, then returns a score for that pair.

Nothing was compressed in advance. The model sees the full question and the full
chunk at once, so one word can change the score.

**Why not use it for everything?** It runs once per question-chunk pair. With 3,000
chunks, one question means 3,000 model runs. Dense search is one model run plus a
lookup.

### So: two stages

Dense search narrows 3,000 chunks to about 30. The reranker puts those 30 in order.

**The consequence to remember:** the reranker only reorders what dense search handed
it. If the right chunk is not in those 30, reranking will never find it.

In [ ]:
QUESTION = "How fast is AI adoption growing across industries?"

# Two numbers per result, from two different models.
#
#   dense   the distance between two vectors that were computed separately and
#           never saw each other
#   rerank  a model that read your question and this chunk TOGETHER
#
# Where they disagree is where the second stage earned its cost. A chunk high on
# rerank and low on dense is one a plain vector search would have ranked poorly.
for rank, hit in enumerate(retrieval.search(index, QUESTION), 1):
    print(f"{rank}. [rerank {hit['rerank']:.3f} | dense {hit['dense']:.3f}] "
          f"p{hit['page']} {hit['content_type']}")
    print(f"   {hit['text'][:150]}\n")

Two score columns. `dense` is the vector distance. `rerank` is the cross-encoder
reading your question and the chunk together.

Where they disagree is where the second stage earned its cost.

In [ ]:
# Did reranking actually change anything here?
candidates = retrieval.retrieve(index, QUESTION, top_k=5)
by_dense = sorted(candidates, key=lambda h: h["dense"], reverse=True)
dense_rank = {h["chunk_id"]: n for n, h in enumerate(by_dense, 1)}

best = dense_rank[candidates[0]["chunk_id"]]
print(f"the best chunk was at dense rank {best}")
print("a plain top-5 would have missed it" if best > 5 else "a plain top-5 would have kept it")

---
## 2. Filtering with metadata

Search finds text that looks similar. Filtering restricts *what it is allowed to look
at* in the first place.

This is the plainest, cheapest thing in the notebook, and every real system does it:

- only documents from this year
- only this customer's documents
- only tables, when the question is about numbers
- only what this user is allowed to see

The filter runs inside the database, so nothing is wasted. That is different from
searching normally and throwing results away afterwards — which would leave a user
with narrow access getting fewer results, silently, instead of their own best five.

Every field you filter on had to be written at ingestion. That is why notebook 1
stored `content_type`, `doc_date`, `page` and `access`. **You cannot filter on
something you did not store**, and adding a field later means re-embedding
everything.

In [ ]:
# Only tables
for hit in retrieval.search(index, "enrolment by site",
                            filters={"content_type": {"$eq": "table_summary"}}):
    print(f"[{hit['rerank']:.3f}] p{hit['page']}  {hit['text'][:80]}")

# Only recent documents
print()
for hit in retrieval.search(index, "adoption trends",
                            filters={"doc_date": {"$gte": "2025"}}):
    print(f"[{hit['rerank']:.3f}] {hit['doc_date']}  p{hit['page']}")

---
## 3. Table rows

In notebook 1, every table went into the index twice: a **summary** in plain English,
and the **raw rows**, linked by a shared `table_id`.

The summary is what search finds, because it contains the words people type. The rows
are what the model needs, because they hold the actual numbers.

This step does the join. When a summary comes back from search, go and fetch that
table's rows too.

Without it, the work in notebook 1 was wasted — you paid for a summary the model
cannot check.

In [ ]:
# The join. `search` returns whatever matched — including a table SUMMARY, which
# matched because it contains the words you typed.
#
# `with_table_rows` then fetches that table's actual rows, by id. Exact, no
# similarity involved: they are found through the shared `table_id` written at
# ingestion, not by searching again.
#
# Read the output in order. The summary should appear immediately before the
# fragments it describes.
hits = retrieval.search(index, "what does the schedule of assessments contain?", top_k=3)

for row in retrieval.with_table_rows(index, hits, DIMS):
    print(f"[{row['position']:>3}] {row['content_type']:<14} p{row['page']}  "
          f"{row['text'][:70]}")

---
## 4. Build the prompt and answer

You have chunks. Now decide what actually goes into the prompt.

There is a token budget. Fill it best-first, so if you run out of room, what gets
dropped is what mattered least.

Then put what survived back into **document order** before sending it. The model
should read passages the way they were written, not the way they were ranked.

This is the least interesting-looking step and one of the most important. Most
"context window" problems are really assembly problems.

In [ ]:
# The whole path: search, rerank, fetch table rows, fill a token budget,
# generate.
#
# `stats` is worth reading next to the answer:
#
#   chunks    how many made it into the prompt
#   tokens    how much of the budget was used
#   dropped   retrieved, then left out because the budget ran out. Anything
#             above 0 was paid for and never seen by the model.
result = retrieval.answer(index, QUESTION, embed_dims=DIMS)

print(result["stats"], "\n")
print(result["answer"])

`dropped` above zero means search returned more than the budget could hold. Those
chunks were paid for and never seen by the model.

**That is the system.** Search, rerank, filter, fetch table rows, fill a budget,
generate.

---
## What is not in this notebook

You will read about these. They are real techniques and they are mostly not worth
their cost, so they are named here rather than built.

**Contextual compression.** Trim each retrieved chunk down to the sentences that
matter, to save tokens. You will see this everywhere, because the popular frameworks
ship it as a one-liner.

In practice it is used far less than it is written about. If your chunks are the
right size there is not much filler to trim. Context windows are large and cheap now.
And the good version costs one LLM call *per chunk*, so five chunks means five extra
calls on every question. Most teams retrieve fewer chunks instead, which is free.

**MMR (Maximal Marginal Relevance).** Drop results that are near-duplicates of ones
already picked, so five slots hold five different facts rather than five wordings of
one. Costs nothing, sometimes helps. Worth trying if you look at your results and see
repetition.

**Multi-query and HyDE.** Ask the same question several ways, or have a model invent
an answer and search for that. Both cost extra calls on every question, and the gains
are inconsistent.

**Neighbour expansion.** Pull in the chunks either side of each result. Useful when
chunks are small and cut mid-argument. At 1024 tokens most chunks already carry their
own context.

### The one that is actually worth adding

**Hybrid search.** Run keyword search (BM25) alongside vector search and combine the
results.

Vector search is weakest at exactly what keyword search is best at: exact terms.
Trial IDs, product codes, error codes, defined terms. Someone searching for
`NCT04368728` wants that document, not documents about similar trials.

It is not here because it needs sparse vectors written during ingestion, so it is an
ingestion change rather than a retrieval one. The index was created with the
`dotproduct` metric specifically so this stays possible.

If you add one thing to this pipeline, add that.